# E-Commerce Lakehouse: Gold Order Summary

## Aggregate items and payments to order level

Calculate merchandise value, freight value, and item count per order.
Separately calculate recorded payment value per order.

Aggregate before joining to prevent multiple item and payment rows
from inflating totals. Merchandise value excludes freight; recorded
payments are kept as a separate metric.

In [ ]:
from pyspark.sql import functions as F

items_df = spark.table("workspace.ecommerce.silver_order_items")
payments_df = spark.table("workspace.ecommerce.silver_payments")

order_item_totals_df = items_df.groupBy("order_id").agg(
    F.count("*").alias("item_count"),
    F.sum("price").alias("merchandise_value"),
    F.sum("freight_value").alias("freight_value")
).withColumn(
    "order_value",
    F.col("merchandise_value") + F.col("freight_value")
)

order_payment_totals_df = payments_df.groupBy("order_id").agg(
    F.count("*").alias("payment_record_count"),
    F.sum("payment_value").alias("recorded_payment_value"),
    F.max(
        F.col("zero_installments_flag").cast("int")
    ).cast("boolean").alias("has_zero_installments"),
    F.max(
        F.col("zero_payment_value_flag").cast("int")
    ).cast("boolean").alias("has_zero_payment"),
    F.max(
        F.col("undefined_payment_type_flag").cast("int")
    ).cast("boolean").alias("has_undefined_payment_type")
)

print("Orders represented in items:", order_item_totals_df.count())
print("Orders represented in payments:", order_payment_totals_df.count())

display(order_item_totals_df.limit(5))
display(order_payment_totals_df.limit(5))

## Combine order-level data

Left join orders to customers, aggregated items, and aggregated payments.
Preserve every order and flag missing item or payment records.

Calculate delivery duration and lateness using calendar dates.
Exclude inconsistent shipping sequences from carrier transit duration.

In [ ]:
orders_df = spark.table("workspace.ecommerce.silver_orders")

customers_df = spark.table(
    "workspace.ecommerce.silver_customers"
).select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
)

gold_order_summary_df = (
    orders_df
    .join(customers_df, on="customer_id", how="left")
    .join(order_item_totals_df, on="order_id", how="left")
    .join(order_payment_totals_df, on="order_id", how="left")
    .withColumn(
        "missing_items_flag",
        F.col("item_count").isNull()
    )
    .withColumn(
        "missing_payments_flag",
        F.col("payment_record_count").isNull()
    )
    .withColumn(
        "purchase_date",
        F.to_date("order_purchase_timestamp")
    )
    .withColumn(
        "purchase_month",
        F.trunc("purchase_date", "month")
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_purchase_timestamp"
        )
    )
    .withColumn(
        "delivery_lateness_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        )
    )
    .withColumn(
        "late_delivery_flag",
        F.when(
            (F.col("order_status") == "delivered")
            & F.col("order_delivered_customer_date").isNotNull()
            & F.col("order_estimated_delivery_date").isNotNull(),
            F.col("delivery_lateness_days") > 0
        )
    )
    .withColumn(
        "carrier_transit_days",
        F.when(
            F.col("shipping_sequence_issue") == False,
            F.datediff(
                "order_delivered_customer_date",
                "order_delivered_carrier_date"
            )
        )
    )
)

gold_order_summary_df.agg(
    F.count("*").alias("order_rows"),
    F.countDistinct("order_id").alias("distinct_order_ids"),
    F.count(
        F.when(F.col("missing_items_flag"), 1)
    ).alias("orders_without_items"),
    F.count(
        F.when(F.col("missing_payments_flag"), 1)
    ).alias("orders_without_payments"),
    F.count(
        F.when(F.col("customer_unique_id").isNull(), 1)
    ).alias("orders_without_customer")
).show(truncate=False)

display(gold_order_summary_df.limit(5))

## Reconcile financial totals and save Gold orders

Compare Gold merchandise, freight, and recorded payment totals
with their Silver sources to detect inflation or loss from joins.

Save the order summary as a replaceable reporting table.
Overwrite is intentional: rerunning this step rebuilds the table
rather than appending duplicate orders.

In [ ]:
source_items = items_df.agg(
    F.sum("price").alias("merchandise"),
    F.sum("freight_value").alias("freight")
).first()

source_payments = payments_df.agg(
    F.sum("payment_value").alias("payments")
).first()

gold_totals = gold_order_summary_df.agg(
    F.sum("merchandise_value").alias("merchandise"),
    F.sum("freight_value").alias("freight"),
    F.sum("recorded_payment_value").alias("payments")
).first()

for metric, source_value in [
    ("merchandise", source_items["merchandise"]),
    ("freight", source_items["freight"]),
    ("payments", source_payments["payments"])
]:
    gold_value = gold_totals[metric]

    print(f"{metric}: source={source_value}, Gold={gold_value}")

    if source_value != gold_value:
        raise ValueError(f"Financial reconciliation failed: {metric}")

table_name = "workspace.ecommerce.gold_order_summary"

(
    gold_order_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_gold_df = spark.table(table_name)

print("Saved Gold order rows:", saved_gold_df.count())